# MATH GR5360 Final Project — Notebook 02

Segmented trend-following strategy notebook. This notebook imports the shared engine, runs one TF sanity check for `Channel WithDDControl`, and then performs the TF walk-forward optimization used in the project-facing workflow.


In [1]:
MARKET_SELECT = 'TY'
QUICK_TEST = True
WALKFORWARD_MODE = 'tf'


In [2]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'mafn_engine').exists() else CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mafn_engine import (
    COLUMBIA_CORE,
    COLUMBIA_NAVY,
    COLUMBIA_RED,
    apply_columbia_theme,
    choose_tf_story_configuration,
    default_tf_grid,
    get_market,
    load_ohlc,
    prepare_analysis_frame,
    run_tf_backtest,
    walk_forward,
)

apply_columbia_theme()
MARKET = get_market(MARKET_SELECT)
DATA_DIR = str(PROJECT_ROOT / 'data')
print(f"Market: {MARKET.ticker} - {MARKET.name} ({MARKET.exchange})")
print(f"PV=${MARKET.PV:,}  Slippage=${MARKET.slpg}  E0=${MARKET.E0:,.0f}")


Market: TY - 10-Year Treasury (CBOT-CME)
PV=$1,000  Slippage=$18.625  E0=$100,000


In [3]:
def _grid_signature(grid: dict[str, np.ndarray]) -> tuple[tuple[str, tuple[float, ...]], ...]:
    signature = []
    for key in sorted(grid):
        values = []
        for value in np.asarray(grid[key]).tolist():
            if isinstance(value, (int, np.integer)):
                values.append(int(value))
            else:
                values.append(float(value))
        signature.append((key, tuple(values)))
    return tuple(signature)


def ensure_analysis_state(force: bool = False) -> None:
    global MARKET, full_df, analysis_df, tf_grid, _analysis_market

    MARKET = get_market(MARKET_SELECT)
    if force or globals().get('_analysis_market') != MARKET_SELECT or 'analysis_df' not in globals():
        full_df = load_ohlc(DATA_DIR, MARKET_SELECT, fallback_synthetic=False)
        analysis_df = prepare_analysis_frame(full_df, MARKET_SELECT)
        _analysis_market = MARKET_SELECT

    tf_grid = default_tf_grid(MARKET_SELECT, quick=QUICK_TEST)


def ensure_walkforward_state(force: bool = False) -> None:
    global wf_bundle, wf_params, wf_equity, wf_ledger, wf_results, _wf_signature

    ensure_analysis_state(force=force)
    signature = (
        MARKET_SELECT,
        WALKFORWARD_MODE,
        bool(QUICK_TEST),
        4,
        1,
        _grid_signature(tf_grid),
    )

    if force or globals().get('_wf_signature') != signature or 'wf_bundle' not in globals():
        wf_bundle = walk_forward(
            analysis_df,
            MARKET_SELECT,
            mode=WALKFORWARD_MODE,
            tf_grid=tf_grid,
            T_years=4,
            tau_quarters=1,
            quick=QUICK_TEST,
            verbose=True,
        )
        _wf_signature = signature

    wf_params = wf_bundle['params']
    wf_equity = wf_bundle['equity']
    wf_ledger = wf_bundle['ledger']
    wf_results = wf_params


ensure_analysis_state(force=True)

tf_combos = len(tf_grid['L']) * len(tf_grid['S'])
print(f"TF grid combinations: {tf_combos:,}")
print('Project-facing workflow: trend-following only.')


✓ Loaded TY from /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5360/Final Project/MATH5360_Final_Project/data/TY-5minHLV.csv
TF grid combinations: 35
Project-facing workflow: trend-following only.


In [4]:
ensure_analysis_state()

story_cfg = choose_tf_story_configuration(MARKET_SELECT, tf_grid=tf_grid, params_df=pd.DataFrame())
tf_sanity = run_tf_backtest(analysis_df, MARKET_SELECT, L=int(story_cfg['L']), S=float(story_cfg['S']))

sanity_summary_df = pd.DataFrame([
    {
        'Family': 'tf',
        'L': int(story_cfg['L']),
        'S': float(story_cfg['S']),
        'Profit': tf_sanity['Profit'],
        'MaxDD': tf_sanity['MaxDD'],
        'Objective': tf_sanity['Objective'],
        'Trades': tf_sanity['NumTrades'],
    },
])
sanity_summary_df


,Family,L,S,Profit,MaxDD,Objective,Trades
0,tf,1440,0.02,56146.1875,24112.8125,2.328479,429


In [5]:
ensure_walkforward_state(force=True)

print()
print('TF WALK-FORWARD COMPLETE')
print('-' * 72)
print(f"Periods: {len(wf_params)}")
print(f"Ledger rows: {len(wf_ledger)}")
wf_params.head()


✓ Loaded TY from /Users/nigelli/Desktop/Columbia MAFN/26Spring/MATH5360/Final Project/MATH5360_Final_Project/data/TY-5minHLV.csv
Walk-Forward [TY] mode=tf: IS=4yr (80,640 bars), OOS=1Q (5,040 bars)
  P1 TF IS_obj=+4.668 OOS_obj=+1.238 votes(tf=19, mr=0)
  P2 TF IS_obj=+5.544 OOS_obj=-0.105 votes(tf=20, mr=0)
  P3 TF IS_obj=+5.526 OOS_obj=+0.728 votes(tf=15, mr=0)
  P4 TF IS_obj=+6.160 OOS_obj=+0.053 votes(tf=16, mr=0)
  P5 TF IS_obj=+6.322 OOS_obj=-0.861 votes(tf=13, mr=0)
  P6 TF IS_obj=+5.362 OOS_obj=-0.417 votes(tf=11, mr=1)
  P7 TF IS_obj=+5.091 OOS_obj=+0.124 votes(tf=12, mr=0)
  P8 TF IS_obj=+5.283 OOS_obj=+0.345 votes(tf=6, mr=0)
  P9 TF IS_obj=+5.366 OOS_obj=-0.813 votes(tf=5, mr=0)
  P10 TF IS_obj=+3.872 OOS_obj=-0.417 votes(tf=4, mr=1)
  P11 TF IS_obj=+4.564 OOS_obj=-0.265 votes(tf=5, mr=0)
  P12 TF IS_obj=+3.725 OOS_obj=-0.404 votes(tf=5, mr=0)
  P13 TF IS_obj=+2.491 OOS_obj=+0.419 votes(tf=3, mr=1)
  P14 TF IS_obj=+2.116 OOS_obj=-0.606 votes(tf=0, mr=3)
  P15 TF IS_obj=+2.4

KeyboardInterrupt: 

In [ ]:
ensure_walkforward_state()

if len(wf_params):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    x = np.arange(len(wf_params))

    axes[0, 0].bar(x - 0.2, wf_params['IS_Profit'] / 1000, width=0.4, color=COLUMBIA_CORE, label='IS')
    axes[0, 0].bar(x + 0.2, wf_params['OOS_Profit'] / 1000, width=0.4, color=COLUMBIA_NAVY, label='OOS')
    axes[0, 0].axhline(0.0, color=COLUMBIA_RED, linewidth=1)
    axes[0, 0].set_title('IS vs OOS Profit')
    axes[0, 0].set_ylabel('Profit ($K)')
    axes[0, 0].legend()

    if 'L' in wf_params:
        axes[0, 1].plot(wf_params['Period'], wf_params['L'], marker='o', color=COLUMBIA_NAVY)
        axes[0, 1].set_title('Selected TF Lookback by Period')
        axes[0, 1].set_ylabel('L (bars)')
        axes[0, 1].set_xlabel('Period')
    else:
        axes[0, 1].text(0.5, 0.5, 'No TF parameter history generated', ha='center', va='center')
        axes[0, 1].set_axis_off()

    axes[1, 0].plot(wf_params['Period'], wf_params['OOS_Objective'], marker='o', color=COLUMBIA_NAVY, label='OOS obj')
    axes[1, 0].plot(wf_params['Period'], wf_params['IS_Objective'], marker='s', color=COLUMBIA_CORE, label='IS obj')
    axes[1, 0].axhline(0.0, color=COLUMBIA_RED, linewidth=1)
    axes[1, 0].set_title('Objective by Period')
    axes[1, 0].legend(loc='best')

    if len(wf_equity):
        axes[1, 1].plot(wf_equity.index, wf_equity['OOS_Equity'], color=COLUMBIA_NAVY)
        axes[1, 1].set_title('Concatenated OOS Equity')
        axes[1, 1].set_ylabel('Equity ($)')
    else:
        axes[1, 1].text(0.5, 0.5, 'No OOS equity generated', ha='center', va='center')
        axes[1, 1].set_axis_off()

    plt.tight_layout()
    plt.show()

wf_params


In [ ]:
ensure_walkforward_state()

if len(wf_ledger):
    print()
    print('OOS TRADE LEDGER PREVIEW')
    print('-' * 72)
    wf_ledger.head(20)
else:
    print('No OOS trades generated.')
